In [ ]:
import random
import time


class UltimateWeightedWheel:  # ultimate weighted wheel game engine with nested probabilities and a rare chance for high-tier rewards
    def __init__(
        self, starting_points=100, starting_spins=5, verbose=True
    ):  # initialize the game with starting points, spins, and verbosity
        self.points = starting_points
        self.free_spins = starting_spins
        self.verbose = verbose  # verbose mode for detailed output
        self.total_spins_cast = 0  # total spins cast counter

        # 1. Core Wheel Segments & Probabilities
        self.wheel_outcomes = [
            "POINTS",
            "MULTIPLIER",
            "FREE_SPINS",
            "DOUBLE",
        ]  # wheel outcomes with nested probabilities
        self.wheel_weights = [
            0.55,
            0.20,
            0.15,
            0.10,
        ]  # wheel weights with nested probabilities

        # 2. Your exact 17 discrete points with an exponentially scaling rarity curve
        self.points_list = [  # points list with exponential scaling
            100,
            150,
            200,
            250,
            300,
            400,
            500,
            750,
            1000,
            1500,
            2000,
            2500,
            3000,
            4000,
            5000,
            7500,
            10000,
        ]
        self.points_weights = [  # points weights with exponential scaling
            0.20,
            0.16,
            0.14,
            0.12,
            0.10,
            0.08,
            0.06,
            0.04,
            0.03,
            0.02,
            0.015,
            0.01,
            0.008,
            0.006,
            0.005,
            0.004,
            0.002,
        ]

        # 3. Your explicit Multiplier nested weights (50% / 30% / 20%)
        self.multiplier_list = [3, 5, 10]  # multiplier list with nested weights
        self.multiplier_weights = [
            0.50,
            0.30,
            0.20,
        ]  # multiplier weights with nested weights

        # 4. Your explicit Free Spins nested weights (50% / 30% / 20%)
        self.spins_payout_list = [2, 3, 5]  # free spins payout list with nested weights
        self.spins_payout_weights = [
            0.50,
            0.30,
            0.20,
        ]  # free spins payout weights with nested weights

    def roll_discrete_points(
        self,
    ):  # rolls a discrete point value from the weighted 17-tier system
        """Picks one exact point value from your weighted 17-tier system."""
        return random.choices(self.points_list, weights=self.points_weights, k=1)[0]

    def roll_nested_multiplier(
        self,
    ):  # rolls a multiplier value from the weighted 3-tier system
        """Picks a multiplier using your explicit 50%/30%/20% configuration."""
        return random.choices(
            self.multiplier_list, weights=self.multiplier_weights, k=1
        )[0]

    def roll_nested_free_spins(
        self,
    ):  # rolls a free spins value from the weighted 3-tier system
        """Picks free spins using your explicit 50%/30%/20% configuration."""
        return random.choices(
            self.spins_payout_list, weights=self.spins_payout_weights, k=1
        )[0]

    def roll_multiplier_base_payout(
        self,
    ):  # rolls a base payout for multiplier outcomes, with a rare chance to hit a higher tier
        """Picks a base payout for multiplier outcomes, with a rare chance to hit a higher tier."""
        rare_high_payout_chance = 0.05  # 5% chance to hit a higher tier
        if (
            random.random() < rare_high_payout_chance
        ):  # if the random number is less than the rare high payout chance, select a random high tier index
            high_tier_start_index = 5  # start index for high tier payouts
            high_tier_count = len(self.points_list) - high_tier_start_index
            random_high_tier_index = high_tier_start_index + random.randrange(
                high_tier_count
            )
            return self.points_list[
                random_high_tier_index
            ]  # randomly select a base payout from the higher tiers (400 to 10,000)
        return random.choice(
            self.points_list[:5]
        )  # randomly select a base payout from the lower tiers (100 to 300)

    def spin(
        self,
    ):  # spins the wheel and processes the outcome, updating points and spins accordingly
        if (
            self.free_spins <= 0
        ):  # if there are no free spins left, print game over message and return False
            print("\n❌ Out of spins! Game Over.")
            print(f"📊 Final Points: {self.points:,}")
            print(f"🎟️ Total Spins Cast: {self.total_spins_cast}")
            return False

        self.free_spins -= 1  # decrement free spins by 1
        self.total_spins_cast += 1  # increment total spins cast by 1

        if self.verbose:
            self.run_spin_animation()

        # 1. Core roll
        landed_outcome = random.choices(
            self.wheel_outcomes, weights=self.wheel_weights, k=1
        )[
            0
        ]  # landed outcome is selected from the wheel outcomes based on their weights

        # 2. Match outcome against the game rules engine
        if (
            landed_outcome == "POINTS"
        ):  # if the landed outcome is "POINTS", roll discrete points and add to total points
            points_won = (
                self.roll_discrete_points()
            )  # roll discrete points from the weighted 17-tier system
            self.points += points_won
            if self.verbose:
                print(
                    f"🎯 Outcome: Standard Points! Added +{points_won:,} to your score."
                )

        elif (
            landed_outcome == "MULTIPLIER"
        ):  # if the landed outcome is "MULTIPLIER", roll a nested multiplier and a base payout, then calculate points won and add to total points
            # Selects your nested multiplier value (3x, 5x, 10x)
            multiplier = (
                self.roll_nested_multiplier()
            )  # rolls a multiplier from the weighted 3-tier system
            # Grabs a reliable base point value from your common lower tiers (100 to 300)
            base_payout = (
                self.roll_multiplier_base_payout()
            )  # rolls a base payout for multiplier outcomes, with a rare chance to hit a higher tier
            points_won = (
                base_payout * multiplier
            )  # calculate points won by multiplying base payout by multiplier
            self.points += points_won
            if self.verbose:
                print(f"🚀 Outcome: Multiplier Bonus Sub-Tier triggered!")
                print(
                    f"    Rolled a [{multiplier}x] Modifier! ({base_payout} Base Points x {multiplier}x = +{points_won:,} points!)"
                )

        elif (
            landed_outcome == "FREE_SPINS"
        ):  # if the landed outcome is "FREE_SPINS", roll nested free spins and add to remaining spins
            # Selects your nested free spins value (2, 3, 5)
            spins_won = (
                self.roll_nested_free_spins()
            )  # rolls free spins from the weighted 3-tier system
            self.free_spins += spins_won
            if self.verbose:
                print(f"🎟️ Outcome: Free Spins Sub-Tier triggered!")
                print(
                    f"    Rolled an Extra Turn token ➡️ Granted +{spins_won} Remaining Spins!"
                )

        elif (
            landed_outcome == "DOUBLE"
        ):  # if the landed outcome is "DOUBLE", double the current points and possibly award an extra spin
            old_points = self.points
            self.points *= 2  # double the current points
            if random.random() < 0.5:  # 50% chance to award an extra spin
                self.free_spins += 1
                if self.verbose:
                    print("🎟️ DOUBLE outcome also awarded an extra spin!")
            if self.verbose:
                print(
                    f"💥 JACKPOT: DOUBLE CURRENT POINTS! Score shifted from {old_points:,} to {self.points:,}!"
                )

        if self.verbose:
            self.display_hud()
        return True

    def run_spin_animation(
        self,
    ):  # runs a simple spinning animation in the console to simulate the wheel spin
        frames = ["◜", "◝", "◞", "◟"]  # frames for the spinning animation
        print("\n🎡 Computing probability layers and spinning... ", end="")
        for _ in range(3):
            for frame in frames:
                print(f"\b{frame}", end="", flush=True)
                time.sleep(0.08)
        print("\b⚡ STOP! ⚡")

    def display_hud(
        self,
    ):  # displays the current points total and remaining spins in a formatted manner
        print("-" * 70)
        print(
            f"📊 POINTS TOTAL: {self.points:,}  |  🎟️ REMAINING SPINS: {self.free_spins}"
        )
        print("-" * 70)


# --- Start Runtime Loop ---
if __name__ == "__main__":
    game = UltimateWeightedWheel(
        starting_points=100, starting_spins=5, verbose=True
    )  # initialize the game with starting points, spins, and verbosity
    if game.verbose:
        print("====== FULLY CALIBRATED MATHEMATICAL REWARDS ENGINE ======")
        game.display_hud()

    while (
        game.free_spins > 0
    ):  # while there are free spins remaining, continue to spin the wheel
        if game.verbose:
            action = (
                input("Press [Enter] to execute a spin (or type 'q' to quit): ")
                .strip()
                .lower()
            )  # prompt the user for input to spin the wheel or quit
            if (
                action == "q"
            ):  # if the user inputs 'q', break the loop and terminate the session
                break
        game.spin()  # spin the wheel and process the outcome
    print(
        f"\n🏁 Session Terminated. Total Accumulated Score: {game.points:,} across {game.total_spins_cast} casts."
    )

====== FULLY CALIBRATED MATHEMATICAL REWARDS ENGINE ======
----------------------------------------------------------------------
📊 POINTS TOTAL: 100  |  🎟️ REMAINING SPINS: 5
----------------------------------------------------------------------

🎡 Computing probability layers and spinning...⚡ STOP! ⚡
🚀 Outcome: Multiplier Bonus Sub-Tier triggered!
    Rolled a [5x] Modifier! (300 Base Points x 5x = +1,500 points!)
----------------------------------------------------------------------
📊 POINTS TOTAL: 1,600  |  🎟️ REMAINING SPINS: 4
----------------------------------------------------------------------

🎡 Computing probability layers and spinning...⚡ STOP! ⚡
🎯 Outcome: Standard Points! Added +200 to your score.
----------------------------------------------------------------------
📊 POINTS TOTAL: 1,800  |  🎟️ REMAINING SPINS: 3
----------------------------------------------------------------------

🎡 Computing probability layers and spinning...⚡ STOP! ⚡
🎟️ Outcome: Free Spins Sub-Tier 

In [ ]:
for _ in range(100):
    wheel = UltimateWeightedWheel(
        starting_points=100, starting_spins=5, verbose=False
    )  # initialize the game with starting points, spins, and verbosity
    while wheel.spin():
        pass
    print(
        f"Wheel Game over -> Points: {wheel.points:,} | Total spins cast: {wheel.total_spins_cast}"
    )  # print the final points and total spins cast after the game is over

    ten_spins = UltimateWeightedWheel(
        starting_points=100, starting_spins=10, verbose=False
    )  # initialize the game with starting points, spins, and verbosity
    while ten_spins.spin():
        pass
    print(
        f"Ten Spins Game over -> Points: {ten_spins.points:,} | Total spins cast: {ten_spins.total_spins_cast}"
    )  # print the final points and total spins cast after the game is over

    zero_starting_points = UltimateWeightedWheel(
        starting_points=0, starting_spins=5, verbose=False
    )  # initialize the game with starting points, spins, and verbosity
    while zero_starting_points.spin():
        pass
    print(
        f"Zero Starting Points Game over -> Points: {zero_starting_points.points:,} | Total spins cast: {zero_starting_points.total_spins_cast}"
    )  # print the final points and total spins cast after the game is over

    ten_spins_zero_points = UltimateWeightedWheel(
        starting_points=0, starting_spins=10, verbose=False
    )  # initialize the game with starting points, spins, and verbosity
    while ten_spins_zero_points.spin():
        pass
    print(
        f"Ten Spins Zero Points Game over -> Points: {ten_spins_zero_points.points:,} | Total spins cast: {ten_spins_zero_points.total_spins_cast}"
    )  # print the final points and total spins cast after the game is over


❌ Out of spins! Game Over.
📊 Final Points: 26,600
🎟️ Total Spins Cast: 5
Wheel Game over -> Points: 26,600 | Total spins cast: 5

❌ Out of spins! Game Over.
📊 Final Points: 20,200
🎟️ Total Spins Cast: 19
Ten Spins Game over -> Points: 20,200 | Total spins cast: 19

❌ Out of spins! Game Over.
📊 Final Points: 4,100
🎟️ Total Spins Cast: 7
Zero Starting Points Game over -> Points: 4,100 | Total spins cast: 7

❌ Out of spins! Game Over.
📊 Final Points: 19,950
🎟️ Total Spins Cast: 16
Ten Spins Zero Points Game over -> Points: 19,950 | Total spins cast: 16

❌ Out of spins! Game Over.
📊 Final Points: 1,650
🎟️ Total Spins Cast: 5
Wheel Game over -> Points: 1,650 | Total spins cast: 5

❌ Out of spins! Game Over.
📊 Final Points: 7,800
🎟️ Total Spins Cast: 10
Ten Spins Game over -> Points: 7,800 | Total spins cast: 10

❌ Out of spins! Game Over.
📊 Final Points: 14,550
🎟️ Total Spins Cast: 11
Zero Starting Points Game over -> Points: 14,550 | Total spins cast: 11

❌ Out of spins! Game Over.
📊 Fina